# RG3 F1: 학습 구간별 순위 변수 축소

순위75 변수10개에서 학습ref만으로 ExtraTrees 중요도를 구해 top3/top6 선택. 각 외부/내부 ref마다 선택을 새로 하고 평가 구간 라벨은 사용하지 않는다. 기본rank75/std4/top3/top6 전문가(각 RF/ET2시드 평균) 단독·쌍평균과 임계값을 내부 F1 선택. 같은7개 관측구간, 정상601·가스19. 새 독립 검증 아닌 개발용 중첩검증.

중요도는 물리적 가스 원인 증거가 아니다. 인덱스0~9는 Hopper,Barrel2,Barrel5,Cushion,PlasticizingTime,pressureSwitchGap,Barrel2std,HopperStd,CushionStd,Barrel5std의 조건부 순위에 대응.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and (y==0).sum()==601
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
memo={};audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];F=z['features_'+n+'_1'];old=z['scores_'+n]
  selector=ExtraTreesClassifier(n_estimators=160,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=42,n_jobs=2).fit(F[ref],y[ref])
  order=np.argsort(-selector.feature_importances_,kind='stable')
  values=[old[:,1]]
  selected={}
  for count in [3,6]:
   ids=order[:count];selected[count]=','.join(map(str,ids));preds=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
     preds.append(m.fit(F[ref][:,ids],y[ref]).predict_proba(F[:,ids])[:,1])
   values.append(np.mean(preds,axis=0))
  values.append(old[:,0])
  memo[key]=np.column_stack(values)
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),top3=selected[3],top6=selected[6],train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
weights=[np.eye(4)[i] for i in range(4)]+[(np.eye(4)[i]+np.eye(4)[j])/2 for i in range(4) for j in range(i+1,4)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b)
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in weights:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_feature_subset',best[5],best[4])]
 for i,name in enumerate(['rank75_reference','rank75_top3','rank75_top6','deep_std4_reference']):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_rank_feature_subset_progress.json').write_text(str(int(b)),encoding='utf-8')
 print('Finished feature-subset block',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_rank_feature_subset_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


Finished feature-subset block 1


Finished feature-subset block 2


Finished feature-subset block 3


Finished feature-subset block 4


Finished feature-subset block 5


Finished feature-subset block 6


Finished feature-subset block 7


                           model       AP  TP  FP  FN  TN       F1   recall
deep_std4_reference_inner_F1_cut 0.079209   4  50  15 551 0.109589 0.210526
inner_F1_selected_feature_subset 0.110096   5  54  14 547 0.128205 0.263158
   rank75_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
        rank75_top3_inner_F1_cut 0.130301   7  54  12 547 0.175000 0.368421
        rank75_top6_inner_F1_cut 0.064849   6 107  13 494 0.090909 0.315789
